In [7]:
import os
import pandas as pd
import numpy as np
import scipy.stats as stats
import seaborn as sns
import matplotlib.pyplot as plt

In [27]:
def merge_features_for_pair(SL_pair: str, base_dir: str = "datasets", feature_dir: str = "feature_output"):
    """
    Merge all computed features (STRING, GTEx, coessentiality, PPI, BioGRID, CGC, essentiality)
    into the main dataset for a given SL_pair.

    Parameters
    ----------
    SL_pair : str
        Pair name, e.g. "ATM_ATR"
    base_dir : str
        Path to datasets directory
    feature_dir : str
        Path to feature_output directory

    Returns
    -------
    main : pd.DataFrame
        Final merged DataFrame with all features.
    """
    # === Load main dataset ===
    main_file = f"{base_dir}/PredictingSLResistanceFeatures_{SL_pair}_main.csv"
    main = pd.read_csv(main_file, low_memory=False)

    # === STRING ===
    biomarker_string = pd.read_csv(f"{feature_dir}/string_score_query_biomarker_{SL_pair}.csv")
    target_string = pd.read_csv(f"{feature_dir}/string_score_query_target1_{SL_pair}.csv")
    main["StringInteractionWithBiomarker"] = biomarker_string["StringInteractionWithBiomarker"].fillna(0) / 1000
    main["StringInteractionWithTarget"] = target_string["StringInteractionWithTarget"].fillna(0) / 1000

    # === GTEx Expression ===
    biomarker_expr = pd.read_csv(f"{feature_dir}/gtex_co_expression_biomarker_query_{SL_pair}.csv")
    target_expr = pd.read_csv(f"{feature_dir}/gtex_co_expression_target1_query_{SL_pair}.csv")
    main["CoexpressionWithBiomarker"] = biomarker_expr["spearman_corr"]
    main["CoexpressionWithTarget"] = target_expr["spearman_corr"]
    main["AvgExpression"] = biomarker_expr["A2_mean_expr"]
    if "A2_expr_variance" in biomarker_expr:
        main["ExpressionVariance"] = biomarker_expr["A2_expr_variance"]

    # === Coessentiality ===
    biomarker_coess = pd.read_csv(f"{feature_dir}/coessentiality_biomarker_query_{SL_pair}.csv")
    target_coess = pd.read_csv(f"{feature_dir}/coessentiality_target1_query_{SL_pair}.csv")
    main["CoessentialityWithBiomarker"] = biomarker_coess["Correlation"]
    main["CoessentialityWithTarget"] = target_coess["Correlation"]
    if "Biomarker_Essentiality_Variance" in biomarker_coess:
        main["EssentialityVariance"] = biomarker_coess["Biomarker_Essentiality_Variance"].fillna(0)
    if "Biomarker_Essentiality_Average" in biomarker_coess:
        main["EssentialityAverage"] = biomarker_coess["Biomarker_Essentiality_Average"].fillna(0)

    # === Shared PPI (BioGRID + STRING) ===
    biogrid_biomarker_ppi = pd.read_csv(f"{feature_dir}/fet_ppi_overlap_biomarker_query_{SL_pair}.csv")
    biogrid_target_ppi = pd.read_csv(f"{feature_dir}/fet_ppi_overlap_target1_query_{SL_pair}.csv")
    string_biomarker_ppi = pd.read_csv(f"{feature_dir}/fet_ppi_overlap_biomarker_query_string_{SL_pair}.csv")
    string_target_ppi = pd.read_csv(f"{feature_dir}/fet_ppi_overlap_target1_query_string_{SL_pair}.csv")

    main["FET_SharedInteractors_Biomarker_BIOGRID"] = biogrid_biomarker_ppi["fet_ppi_overlap"]
    main["FET_SharedInteractors_Target_BIOGRID"] = biogrid_target_ppi["fet_ppi_overlap"]
    main["FET_SharedInteractors_Biomarker_STRING"] = string_biomarker_ppi["fet_ppi_overlap"]
    main["FET_SharedInteractors_Target_STRING"] = string_target_ppi["fet_ppi_overlap"]

    # === BioGRID Physical interactions ===
    biogrid_biomarker_interaction = pd.read_csv(f"{feature_dir}/biogrid_biomarker_query_{SL_pair}.csv")
    biogrid_target_interaction = pd.read_csv(f"{feature_dir}/biogrid_target1_query_{SL_pair}.csv")
    main["BIOGRIDPhysicalInteractionQueryBiomarker"] = biogrid_biomarker_interaction["BIOGRIDPhysicalInteractionQueryBiomarker"]
    main["BIOGRIDPhysicalInteractionQueryTarget"] = biogrid_target_interaction["BIOGRIDPhysicalInteractionQueryTarget1"]

    # === Biomarker Type (TSG/Oncogene) ===
    biomarker_type = pd.read_csv(f"{feature_dir}/biomarker_type_{SL_pair}.csv")
    main["BiomarkerType"] = biomarker_type["TSG_Label"]

    # === Essentiality Percentage ===
    essentiality_percentage = pd.read_csv(f"{feature_dir}/essentiality_percentage_per_gene.csv")
    main = main.merge(
        essentiality_percentage,
        left_on="entrez_id_query",
        right_on="entrez_id",
        how="left"
    ).drop(columns=["entrez_id"])
    main["Essentiality_Percentage"] = main["Essentiality_Percentage"].fillna(0)
    main = main.dropna(subset=['ensembl_gene_id_query']).reset_index(drop=True)
    main= main.dropna(subset=['CoexpressionWithBiomarker','CoexpressionWithTarget']).reset_index(drop=True)

    # === Save final merged dataset ===
    output_file = f"{base_dir}/PredictingSLResistanceFeatures_{SL_pair}_main_withfeatures.csv"
    main.to_csv(output_file, index=False)

    print(f"✅ Final merged dataset saved for {SL_pair}")
    return main


In [28]:
# === Config ===
excel_file = "biomarker_target_genes.xlsx"
pairs_df = pd.read_excel(excel_file)

for _, row in pairs_df.iterrows():
    biomarker = row["Biomarker"]
    target1 = row["Target"]

    SL_pair = f"{biomarker}_{target1}"

    merge_features_for_pair(SL_pair)


✅ Final merged dataset saved for MSH6_WRN
✅ Final merged dataset saved for CDH1_ROS1
✅ Final merged dataset saved for ATM_ATR
✅ Final merged dataset saved for BRCA1_ATR
✅ Final merged dataset saved for CCNE1_PKMYT1
✅ Final merged dataset saved for FBXW7_PKMYT1
✅ Final merged dataset saved for BRCA1_PARG
✅ Final merged dataset saved for BRCA1_USP1
✅ Final merged dataset saved for BRCA1_POLQ
✅ Final merged dataset saved for KRAS_PLK1
